# PROTEIN GM AD

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
from statsmodels.stats.multitest import multipletests
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict



#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors
    
def pval_print(pval, sci_thresh=1e-3, decimals=3):
    if pval == 0:
        return "0"
    if pval < sci_thresh:
        return f"{pval:.{decimals}e}"
    return f"{pval:.{decimals}f}"
        
import pandas as pd

def tukey_to_df(res, labels):
    

    ci = res.confidence_interval()
    rows = []

    for i in range(len(labels)):
        for j in range(i+1, len(labels)):
            rows.append({
                "group1": labels[i],
                "group2": labels[j],

                # Tukey q statistic
                "statistic": res.statistic[i, j],

                # mean difference
                "mean_diff": res.statistic[i, j],

                # adjusted p-value
                "p_adj": res.pvalue[i, j],

                # confidence intervals
                "ci_low": ci.low[i, j],
                "ci_high": ci.high[i, j]
            })

    return pd.DataFrame(rows)

# load in sample all cells adata

In [ ]:
adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')
adata = adata[adata.obs.leiden != 'default']

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}


cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))

adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)




adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'Oligos', 'dense aBeta', 'diffuse aBeta', 'all aBeta']
clustering_column = 'leiden'
adata.obsm['X_spatial'] = adata.obs[['spatial_X','spatial_Y']].values


for i,feat in enumerate(features):
    print(feat)
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    ss = adata[(adata.obs.ImageType =='AD')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    #ss = ss[ss.obs[dt] < 200]

    
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_AD_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_AD_GM.svg')
   

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    
    
    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    
    
    
    
    
    
    
    
    




    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_AD_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_AD_GM.svg', format = 'svg')
    
    
    
    
    
    oDFpremean = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index()
    oDFcount = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['ImageID', clustering_column]).sum().unstack()

    oDFpremean_totals = []

    for ii in oDFpremean.index:
        col = oDFpremean.loc[ii]
        samp = col['ImageID']
        clus = col[clustering_column]
        oDFpremean_totals.append(oDFcount.loc[samp][0][clus])

    oDFpremean['total_in_samp'] = oDFpremean_totals
    oDFpremean['props_in_samp'] = oDFpremean[0]/oDFpremean['total_in_samp']
    oDFpremean = oDFpremean.drop([0, 'total_in_samp'], axis = 1)

    oDF_normed = oDFpremean.groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDF_normed_pct = 100*oDF_normed/oDF_normed.sum(axis=0)
    
    
    oDF_normed_pct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm) (normed)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    
    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_AD_GM_abunnormed.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_AD_GM_abunnormed.svg', format = 'svg')
    
    
    
    
    #ss = adataScaled[(adataScaled.obs.ImageType =='AD')]
    #ss = ss[ss.obs.Parent=='Grey Matter']
    IMList = ['3196','2997','3155','3280','1796']

    for im in IMList:
        print(dt)
        sss = adata[(adata.obs.ImageIDOLD.isin([im]))]
        from matplotlib.colors import Colormap as CM
        from pylab import *
        #cmap = cm.get_cmap('gist_ncar', len(BINS) + 4)
        cmap = cm.get_cmap('nipy_spectral', len(BINS) + 2)   
        cList = [matplotlib.colors.rgb2hex(cmap(i)) for i in range(cmap.N)]
        cList = cList[::-1]
        cList = cList[2:]

        out = pd.cut(sss.obs[dt].clip(0,499),bins=BINS, right=True)
        sss.obs['Quantile'] = out.values
        sss.obs['QuantileC'] = out.cat.codes.values
        print(cList)
        fig,ax=plt.subplots(1,1,figsize=(7,7))
        sc.pl.scatter(sss, basis='spatial',color=['Quantile'],ax=ax,size=15.0,alpha=1,palette=cList,
                          show=False, title=sss.obs.ImageIDOLD[0] + ' ' + feat)
        
        # plot only first bin as stars
        
        firstbin = sss[sss.obs.Quantile == sss.obs['Quantile'].cat.categories[0]]
        
        plt.scatter(x = firstbin.obs.spatial_X, y = firstbin.obs.spatial_Y, c = cList[0], marker="*", s=15)
        
        plt.tight_layout()
        
        plt.savefig('figures/' + feat + '_' + im + '_distance_map.tiff', format = 'tiff')
        plt.savefig('figures/' + feat + '_' + im + '_distance_map.svg', format = 'svg')
        
        #plt.savefig(f'finalfiguresv4/{im}_Distance50um.png', dpi=300)


# stats testing

In [ ]:
def plot_cdf_by_category(df, category_col, value_col, palette=None, title = 'CDF plot by category'):
    """
    Plots CDF lines color-coded by category with a custom color palette.

    Parameters:
    df (pd.DataFrame): DataFrame containing the data.
    category_col (str): Name of the column containing categories.
    value_col (str): Name of the column containing values.
    palette (dict or list, optional): Custom color palette. If a dict, it should map categories to colors.
                                      If a list, it should contain colors in the order of categories in the DataFrame.

    Returns:
    None
    """
    plt.figure(figsize=(10, 6))
    
    categories = df[category_col].unique()
    if palette is None:
        palette = sns.color_palette(n_colors=len(categories))
    elif isinstance(palette, list):
        if len(palette) < len(categories):
            raise ValueError("Palette list must have at least as many colors as there are categories")
        palette = dict(zip(categories, palette))

    for category in categories:
        subset = df[df[category_col] == category]
        sorted_values = np.sort(subset[value_col])
        cdf_values = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
        
        plt.plot(sorted_values, cdf_values, marker='.', linestyle='none', label=category, color=palette[category])
    
    plt.title(title)
    plt.xlabel("Value")
    plt.ylabel("Cumulative Probability")
    plt.legend(title=category_col)
    plt.show()


In [ ]:


import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

clustering_column = 'leiden'

tukeyresvals = []
anova_pvals = []
anova_test_stats = []

for i,feat in enumerate(features):
    
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata[(adata.obs.ImageType =='AD')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    #ss = ss[ss.obs[dt] < 200]
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    lowest_mean_group = groups[means.index(np.min(means))]
    
    f_statistic, p_value = f_oneway(*group_data)
    
    
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
    print(tukey_results)
    
    
    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    p = tukey_results.plot_simultaneous(comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um')
    p.savefig('figures/' + feat + '_tukeyplot_AD_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_tukeyplot_AD_GM.svg', format = 'svg')
    
    
    
    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=groups[i], linewidth=2, color=cMapDict[groups[i]])
        


    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title(feat)
    plt.legend()
    plt.show()
    
    
    plot_cdf_by_category(anova_table, 'leiden', dt, palette=cMapDict, title=feat)
    

adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_tukeyplot_AD_GM.csv')
    
    
    
    





### ADDY PER SAMPLE COMPARISON

In [ ]:

import sys
import os
import anndata 
import pandas as pd
import scanpy as sc
import seaborn as sns; sns.set(color_codes=True)




adata = anndata.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')
adata = adata[adata.obs.leiden != 'default']
adata = adata[adata.obs.Parent == 'Grey Matter']

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}


cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))

adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)


features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'Oligos', 'dense aBeta', 'diffuse aBeta', 'all aBeta']
clustering_column = 'leiden'

adata_meta = adata.obs

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774







In [ ]:
import matplotlib.pyplot as plt
def plot_microglial_clusters(ss, dt, title = None, prop = False, savepath = False, return_data = True):
    ## Plot AD statistics
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    ax = plt.figure(figsize=(8, 6))
    matpl=sc.pl.matrixplot(ss, var_names = ss.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                 dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)
    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})
    oDF = ss.obs.groupby(['Quantile','leiden','ImageID']).size().T.reset_index().groupby(['Quantile','leiden']).mean().unstack(level=1).T.reset_index(drop=True, level = 0)
    
    if prop:
        oDFPct = 100*oDF/oDF.sum(axis=0)
    else:
        oDFPct =oDF
    
    
    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    outputDF = oDFPct.copy()
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.ylabel('# of \n Microglial cells')
    plt.xlabel('')
    plt.title(title)
    
    if savepath:
        plt.savefig(savepath)
    else:
        plt.show()
    
    
    #oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()
#     oDFPct = oDF.T#/oDF.sum(axis=1)
#     oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
#     ax[1].set_ylabel('% of cells that \n are Microglial clusters')
#     ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
#     ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)

#     ax[1].set_ylim(0,max(50,oDFPct.max()))
#     plt.tight_layout()
    if return_data:
        
        return outputDF
# set colors for leiden clusters
cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


In [ ]:

memetime = []
features = ['custom_distance_to_nearest_vasculature_um', 'custom_distance_to_nearest_ApoE_um', 'custom_distance_to_nearest_aBeta_um', 'fabseg_distance_to_nearest_Astrocytes_um', 'fabseg_distance_to_nearest_Neurons_um', 'fabseg_distance_to_nearest_Oligodendrocytes_um', 'fabseg_distance_to_nearest_dense_aBeta_um', 'fabseg_distance_to_nearest_diffuse_aBeta_um']
from scipy.stats import chisquare

clusters = adata.obs.leiden.unique()
distances = ['(0, 5]', '(5, 10]', '(10, 15]', '(15, 20]', '(20, 30]', '(30, 50]',
       '(50, 75]', '(75, 100]', '(100, 200]', '(200, 500]']

samples_t = []
clusters_t = []
features_t = []
chi_t = []
pvals_t = []


samples2_t = []
features2_t = []
distances2_t = []
chi2_t = []
pvals2_t = []

for feat in features:
    allAD = adata[adata.obs['ImageType']=='AD']
    expectedDF = plot_microglial_clusters(allAD, dt = feat, title = feat + ' all samples')
    



    outDict =  {}
    for sID in adata.obs['ImageID'].unique():
        if 'Ctl' in sID:
            continue
        ss = adata[adata.obs['ImageID']==sID]
        outDict[sID] = plot_microglial_clusters(ss, dt = feat, title = feat + ' ' + sID)
        plot_microglial_clusters(ss, dt = feat, title = feat + ' ' + sID, return_data=False, prop=False, savepath='figures/' + feat + ' ' + sID + '_unprop.svg')
        plot_microglial_clusters(ss, dt = feat, title = feat + ' ' + sID, return_data=False, prop=True, savepath='figures/' + feat + ' ' + sID + '_prop.svg')

        
    # X2 per cluster
        
    for cluster in clusters:
        
        print('----------------------------------------------------------------')
        expectedDF.columns = expectedDF.columns.astype(str)
        cId = cluster
        for sID in outDict.keys():
            outDict[sID].columns = outDict[sID].columns.astype(str)
            #normalize
            fObs = 100*outDict[sID].loc[cId]/outDict[sID].loc[cId].sum()
            #normalize
            fExp = 100*expectedDF.loc[cId]/expectedDF.loc[cId].sum()
            
            fObs_ids_equal_to_0 = fObs == 0
            fExp_ids_equal_to_0 = fExp == 0 
            
            fObs = fObs[fExp_ids_equal_to_0 == False]
            fExp = fExp[fExp_ids_equal_to_0 == False]
            
            
            
            chi_stat, p_val = chisquare(fObs,
                                        fExp, axis=None)
            
            
            print(f"For feature {feat} and sample {sID} and cluster {cId}, Chi-square statistic = {chi_stat}, p-value = {p_val}")
            samples_t.append(sID)
            clusters_t.append(cId)
            features_t.append(feat)
            chi_t.append(chi_stat)
            pvals_t.append(p_val)
        
        
        
        
    for distance in distances:
        
        print('-------------------------------------------------------------')
        ## Compute per distance
        expectedDF.columns = expectedDF.columns.astype(str)
        #only farthest cells

        colID = distance

        for sID in outDict.keys():
            #normalize obsVal 
            obsVal = 100*outDict[sID].loc[:,colID]/ outDict[sID].loc[:,colID].sum()
            expVal = 100*expectedDF.loc[:,colID]/ expectedDF.loc[:,colID].sum()


    #             obsVal_ids_equal_to_0 = obsVal == 0
    #             expVal_ids_equal_to_0 = expVal == 0 

    #             obsVal = obsVal[obsVal_ids_equal_to_0 == False]
    #             expVal = expVal[expVal_ids_equal_to_0 == False]


            groups_expected_to_be_represented = expVal.index[expVal != 0]
            groups_expected_to_not_be_represented = expVal.index[expVal == 0]

            memetime.append(groups_expected_to_not_be_represented)


            obsVal = obsVal[groups_expected_to_be_represented]
            expVal = expVal[groups_expected_to_be_represented]
            
            chi_stat, p_val = chisquare(obsVal, 
                                        f_exp=expVal, axis=None)

            # if np.isnan(chi_stat):
            #     fdjklfd


            print(f"For feature {feat} and sample {sID} at distance {colID}, Chi-square statistic = {chi_stat}, p-value = {p_val}")

            samples2_t.append(sID)
            features2_t.append(feat)
            distances2_t.append(colID)
            chi2_t.append(chi_stat)
            pvals2_t.append(p_val)
      
    
t1 = pd.DataFrame({'sample':samples_t, 'cluster':clusters_t, 'feature':features_t, 'chi_statistic':chi_t, 'p_value':pvals_t})
t2 = pd.DataFrame({'sample':samples2_t, 'distance':distances2_t, 'feature':features2_t, 'chi_statistic':chi2_t, 'p_value':pvals2_t})


t1.to_csv('figures/chi_squared_distances_results.csv')
t2.to_csv('figures/chi_squared_distances_results_SPECIFIC_DISTANCE.csv')

# PROTEIN GM CONTROL

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# load in sample all cells adata

In [ ]:
adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')
adata = adata[adata.obs.leiden != 'default']

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}


cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))

adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)




adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes']
feature_names = ['custom_vasc', 'Astrocytes', 'Neurons', 'Oligos']
clustering_column = 'leiden'

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    ss = adata[(adata.obs.ImageType =='Ctl')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_Ctl_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_Ctl_GM.svg')
   

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    



    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_Ctl_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_Ctl_GM.svg', format = 'svg')


# stats testing

In [ ]:


import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons']
feature_names = ['custom_vasc', 'Astrocytes', 'Neurons']

clustering_column = 'leiden'

tukeyresvals = []
anova_pvals = []
anova_test_stats = []

for i,feat in enumerate(features):
    
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata[(adata.obs.ImageType =='Ctl')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    lowest_mean_group = groups[means.index(np.min(means))]
    
    f_statistic, p_value = f_oneway(*group_data)
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
    print(tukey_results)
    
    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    p = tukey_results.plot_simultaneous(comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um')
    p.savefig('figures/' + feat + '_tukeyplot_Ctl_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_tukeyplot_Ctl_GM.svg', format = 'svg')
    
    
    
    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Density Distributions")
    plt.legend()
    plt.show()
    
    
adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_tukeyplot_Ctl_GM.csv')
    
    







# PROTEIN ALL SAMPLES

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# load in sample all cells adata

In [ ]:
adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')
adata = adata[adata.obs.leiden != 'default']

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}


cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))

adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)




adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes']
feature_names = ['custom_vasc', 'Astrocytes', 'Neurons', 'Oligos']
clustering_column = 'leiden'

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    #ss = adata[(adata.obs.ImageType =='Ctl')]
    ss = adata[adata.obs.Parent=='Grey Matter']
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_allsamples_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_allsamples_GM.svg')
   

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    



    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_allsamples_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_allsamples_GM.svg', format = 'svg')


# stats testing

In [ ]:


import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons']
feature_names = ['custom_vasc', 'Astrocytes', 'Neurons']

clustering_column = 'leiden'

tukeyresvals = []
anova_pvals = []
anova_test_stats = []

for i,feat in enumerate(features):
    
   
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata[adata.obs.Parent=='Grey Matter']
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    lowest_mean_group = groups[means.index(np.min(means))]
    
    f_statistic, p_value = f_oneway(*group_data)
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
    print(tukey_results)
    
    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    p = tukey_results.plot_simultaneous(comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um')
    p.savefig('figures/' + feat + '_tukeyplot_allsamples_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_tukeyplot_allsamples_GM.svg', format = 'svg')
    
    
    
    if feat == 'custom_distance_to_nearest_vasculature':
        p = tukey_results.plot_simultaneous(comparison_name='BLM', figsize=(7,5), xlabel=feat + '_um')
        p.savefig('figures/' + feat + '_tukeyplot_allsamples_GM_BLM_focus.tiff', format = 'tiff')
        p.savefig('figures/' + feat + '_tukeyplot_allsamples_GM_BLM_focus.svg', format = 'svg')
        
    
    
    
    
    
    
    
    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Density Distributions")
    plt.legend()
    plt.show()
    
    
    
    
    
adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_tukeyplot_allsamples_GM_BLM_focus.csv')






# PROTEIN ONLY 1796lowest_mean_group

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# load in sample all cells adata

In [ ]:
adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')
adata = adata[adata.obs.leiden != 'default']

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}


cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))

adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)




adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']
clustering_column = 'leiden'

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    ss = adata[(adata.obs.ImageID == 'Ctl-3')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_1796_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_violin_1796_GM.svg')
   

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    



    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_1796_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_1796_GM.svg', format = 'svg')


# stats testing

In [ ]:


import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

clustering_column = 'leiden'

tukeyresvals = []
anova_pvals = []
anova_test_stats = []


for i,feat in enumerate(features):
    
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata[(adata.obs.ImageID == 'Ctl-3')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    lowest_mean_group = groups[means.index(np.min(means))]
    
    f_statistic, p_value = f_oneway(*group_data)
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
    print(tukey_results)
    
    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    p = tukey_results.plot_simultaneous(comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um')
    p.savefig('figures/' + feat + '_tukeyplot_1796_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_tukeyplot_1796_GM.svg', format = 'svg')
    
    
    
    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Density Distributions")
    plt.legend()
    plt.show()
    
    
    
    
adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_tukeyplot_1796_GM.csv')







# MORPH

# load in sample all cells adata

In [ ]:
adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('final_h5ads/clustering_and_morph_clustering.h5')

change_cluster_names_dict_morph = {
    '1': 'Rounded',
    '2': 'Intermediate',
    '0_ram':'Ramified 2',
    '1_ram':'Ramified 3',
    '2_ram':'Ramified 1'
}

cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

new_name_order_morph = ['Rounded', 'Intermediate', 'Ramified 1', 'Ramified 2', 'Ramified 3']
colors_in_order_morph = [cMapDict_morph[i] for i in new_name_order_morph]
cMapDict_morph = dict(zip(new_name_order_morph, colors_in_order_morph))

adata.obs['final_leiden_M1_updated'] = [change_cluster_names_dict_morph[i] for i in adata.obs.final_leiden_M1]
adata.obs['final_leiden_M1_updated'] = pd.Categorical(adata.obs['final_leiden_M1_updated'], categories=new_name_order_morph, ordered=False)



adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }






# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'Oligos', 'dense aBeta', 'diffuse aBeta', 'all aBeta']
clustering_column = 'final_leiden_M1_updated'

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    ss = adata[(adata.obs.ImageType =='AD')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict_morph, save= feat + '_violin_MORPH_AD_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict_morph, save= feat + '_violin_MORPH_AD_GM.svg')

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict_morph, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)




    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_MORPH_AD_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_MORPH_AD_GM.svg', format = 'svg')
    
    
    
    
    oDFpremean = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index()
    oDFcount = ss.obs.groupby(['Quantile',clustering_column,'ImageID']).size().T.reset_index().groupby(['ImageID', clustering_column]).sum().unstack()

    oDFpremean_totals = []

    for ii in oDFpremean.index:
        col = oDFpremean.loc[ii]
        samp = col['ImageID']
        clus = col[clustering_column]
        oDFpremean_totals.append(oDFcount.loc[samp][0][clus])

    oDFpremean['total_in_samp'] = oDFpremean_totals
    oDFpremean['props_in_samp'] = oDFpremean[0]/oDFpremean['total_in_samp']
    oDFpremean = oDFpremean.drop([0, 'total_in_samp'], axis = 1)

    oDF_normed = oDFpremean.groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDF_normed_pct = 100*oDF_normed/oDF_normed.sum(axis=0)
    
    
    oDF_normed_pct.T.plot(kind='bar',stacked=True,color=cMapDict_morph, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm) (normed)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    
    plt.tight_layout()
    plt.savefig('figures/' + feat + '_barplot_MORPH_AD_GM_abunnormed.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_barplot_MORPH_AD_GM_abunnormed.svg', format = 'svg')


# stats testing

In [ ]:


import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

clustering_column = 'final_leiden_M1_updated'


tukeyresvals = []
anova_pvals = []
anova_test_stats = []

for i,feat in enumerate(features):
    
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata[(adata.obs.ImageType =='AD')]
    ss = ss[ss.obs.Parent=='Grey Matter']
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    lowest_mean_group = groups[means.index(np.min(means))]
    
    f_statistic, p_value = f_oneway(*group_data)
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
    print(tukey_results)
    
    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    p = tukey_results.plot_simultaneous(comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um')
    p.savefig('figures/' + feat + '_tukeyplot_MORPH_AD_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_tukeyplot_MORPH_AD_GM.svg', format = 'svg')
    
    
    
    
    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Density Distributions")
    plt.legend()
    plt.show()
    
    

    
    
adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_tukeyplot_MORPH_AD_GM.csv')






# NEIGHBOR GM AD

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# load in sample all cells adata

In [ ]:

adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('MG_neighborhood/sub_obj_with_spatial_clustering.h5ad')


# adata = adata[adata.obs.leiden != 'default']

# new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



# change_cluster_names_dict = {'MG_0': 'Microglia 1',
#          'MG_1': 'Microglia 2',
#          'MO_MAC_0': 'Monocytes',
#          'MO_MAC_1': 'PVM',
#          'MO_MAC_2': 'BLM',
#          'default': 'default'
# }


# cMapDict = {'Microglia 1': '#e377c2',
#  'Microglia 2': '#b5bd61',
#  'Monocytes': '#d62728',
#  'PVM': '#17becf',
#  'BLM': '#ff7f0e',
#  'default': '#8c564b'
# }


# colors_in_order = [cMapDict[i] for i in new_name_order]
# cMapDict = dict(zip(new_name_order, colors_in_order))

# adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

# adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)


cMapDict  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }

adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'Oligos', 'dense aBeta', 'diffuse aBeta', 'all aBeta']
clustering_column = 'spatial_clustering'

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    ss = adata
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_nhood_clusters_violin_AD_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_nhood_clusters_violin_AD_GM.svg')
   

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'origfilenum']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    



    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.savefig('figures/' + feat + '_nhood_clusters_barplot_AD_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_nhood_clusters_barplot_AD_GM.svg', format = 'svg')


# stats testing

In [ ]:
def plot_simultaneous_custom(testo, comparison_name=None, ax=None, figsize=(10,6),
                          xlabel=None, ylabel=None, neworder = 'he'):
        from statsmodels.graphics import utils

        """Plot a universal confidence interval of each group mean

        Visualize significant differences in a plot with one confidence
        interval per group instead of all pairwise confidence intervals.

        Parameters
        ----------
        comparison_name : str, optional
            if provided, plot_intervals will color code all groups that are
            significantly different from the comparison_name red, and will
            color code insignificant groups gray. Otherwise, all intervals will
            just be plotted in black.
        ax : matplotlib axis, optional
            An axis handle on which to attach the plot.
        figsize : tuple, optional
            tuple for the size of the figure generated
        xlabel : str, optional
            Name to be displayed on x axis
        ylabel : str, optional
            Name to be displayed on y axis

        Returns
        -------
        Figure
            handle to figure object containing interval plots

        Notes
        -----
        Multiple comparison tests are nice, but lack a good way to be
        visualized. If you have, say, 6 groups, showing a graph of the means
        between each group will require 15 confidence intervals.
        Instead, we can visualize inter-group differences with a single
        interval for each group mean. Hochberg et al. [1] first proposed this
        idea and used Tukey's Q critical value to compute the interval widths.
        Unlike plotting the differences in the means and their respective
        confidence intervals, any two pairs can be compared for significance
        by looking for overlap.

        References
        ----------
        .. [*] Hochberg, Y., and A. C. Tamhane. Multiple Comparison Procedures.
               Hoboken, NJ: John Wiley & Sons, 1987.

        Examples
        --------
        >>> from statsmodels.examples.try_tukey_hsd import cylinders, cyl_labels
        >>> from statsmodels.stats.multicomp import MultiComparison
        >>> cardata = MultiComparison(cylinders, cyl_labels)
        >>> results = cardata.tukeyhsd()
        >>> results.plot_simultaneous()
        <matplotlib.figure.Figure at 0x...>

        This example shows an example plot comparing significant differences
        in group means. Significant differences at the alpha=0.05 level can be
        identified by intervals that do not overlap (i.e. USA vs Japan,
        USA vs Germany).

        >>> results.plot_simultaneous(comparison_name="USA")
        <matplotlib.figure.Figure at 0x...>

        Optionally provide one of the group names to color code the plot to
        highlight group means different from comparison_name.
        """
        fig, ax1 = utils.create_mpl_ax(ax)
        if figsize is not None:
            fig.set_size_inches(figsize)
        if getattr(testo, 'halfwidths', None) is None:
            testo._simultaneous_ci()
        means = testo._multicomp.groupstats.groupmean
        print(len(means))
        
        means = means[neworder]


        sigidx = []
        nsigidx = []
        minrange = [means[i] - testo.halfwidths[neworder][i] for i in range(len(means))]
        maxrange = [means[i] + testo.halfwidths[neworder][i] for i in range(len(means))]

        if comparison_name is None:
            ax1.errorbar(means, lrange(len(means)), xerr=testo.halfwidths[neworder],
                         marker='o', linestyle='None', color='k', ecolor='k')
        else:
            if comparison_name not in testo.groupsunique[neworder]:
                raise ValueError('comparison_name not found in group names.')
            midx = np.where(testo.groupsunique[neworder]==comparison_name)[0][0]
            for i in range(len(means)):
                if testo.groupsunique[neworder][i] == comparison_name:
                    continue
                if (min(maxrange[i], maxrange[midx]) -
                                         max(minrange[i], minrange[midx]) < 0):
                    sigidx.append(i)
                else:
                    nsigidx.append(i)
            #Plot the main comparison
            ax1.errorbar(means[midx], midx, xerr=testo.halfwidths[neworder][midx],
                         marker='o', linestyle='None', color='b', ecolor='b')
            ax1.plot([minrange[midx]]*2, [-1, testo._multicomp.ngroups],
                     linestyle='--', color='0.7')
            ax1.plot([maxrange[midx]]*2, [-1, testo._multicomp.ngroups],
                     linestyle='--', color='0.7')
            #Plot those that are significantly different
            if len(sigidx) > 0:
                ax1.errorbar(means[sigidx], sigidx,
                             xerr=testo.halfwidths[neworder][sigidx], marker='o',
                             linestyle='None', color='r', ecolor='r')
            #Plot those that are not significantly different
            if len(nsigidx) > 0:
                ax1.errorbar(means[nsigidx], nsigidx,
                             xerr=testo.halfwidths[neworder][nsigidx], marker='o',
                             linestyle='None', color='0.5', ecolor='0.5')

        ax1.set_title('Multiple Comparisons Between All Pairs (Tukey)')
        r = np.max(maxrange) - np.min(minrange)
        ax1.set_ylim([-1, testo._multicomp.ngroups])
        ax1.set_xlim([np.min(minrange) - r / 10., np.max(maxrange) + r / 10.])
        ylbls = [""] + testo.groupsunique.astype(str)[neworder].tolist() + [""]
        ax1.set_yticks(np.arange(-1, len(means) + 1))
        ax1.set_yticklabels(ylbls)
        print('ji')
        ax1.set_xlabel(xlabel if xlabel is not None else '')
        ax1.set_ylabel(ylabel if ylabel is not None else '')
        return fig

In [ ]:


import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

clustering_column = 'spatial_clustering'

tukeyresvals = []
anova_pvals = []
anova_test_stats = []


for i,feat in enumerate(features):
    
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata
    # ss = adata[(adata.obs.ImageType =='AD')]
    # ss = ss[ss.obs.Parent=='Grey Matter']
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    
    lowest_mean_group = groups[means.index(np.min(means))]
    
    f_statistic, p_value = f_oneway(*group_data)
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])

    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    #tukey_results.groupsunique = np.array(['1', '0', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4','5', '6', '7', '8', '9'])
    
    #p = plot_simultaneous_custom(tukey_results, comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um', neworder = [0,1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
    p = plot_simultaneous_custom(tukey_results, comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um', neworder = [7, 6, 5, 4, 3, 2, 15, 14, 13, 12, 11, 10, 9, 8, 1, 0])
   
    p.savefig('figures/' + feat + '_nhood_clusters_tukeyplot_AD_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_nhood_clusters_tukeyplot_AD_GM.svg', format = 'svg')
    
    
    
    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Density Distributions")
    plt.legend()
    plt.show()
    
    
    
   
adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_nhood_clusters_tukeyplot_AD_GM.csv')







In [ ]:


# import pandas as pd
# from scipy.stats import f_oneway
# from statsmodels.stats.multicomp import pairwise_tukeyhsd

# features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
# feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

# clustering_column = 'spatial_clustering'




# for i,feat in enumerate(features):
    
    
    
#     #1 px=0.3774um
    
#     adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
#     dt = feat + '_um'
#     ss = adata
#     # ss = adata[(adata.obs.ImageType =='AD')]
#     # ss = ss[ss.obs.Parent=='Grey Matter']
    
#     anova_table = ss.obs[[clustering_column, feat + '_um']]
    
#     groups = anova_table[clustering_column].unique()
    
    
#     group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
#     group_dict = dict(zip(groups, group_data))
    
#     means = [np.mean(i) for i in group_data]
#     meantable = pd.DataFrame({'names':groups, 'means':means})
    
#     lowest_mean_group = groups[means.index(np.min(means))]
    
#     f_statistic, p_value = f_oneway(*group_data)
#     print(feat)
#     print(p_value)
    
#     tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])

    
#     #tukey_results.groupsunique = np.array(['1', '0', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4','5', '6', '7', '8', '9'])
    
#     #p = plot_simultaneous_custom(tukey_results, comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um', neworder = [0,1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
#     p = plot_simultaneous_custom(tukey_results, comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um', neworder = [7, 6, 5, 4, 3, 2, 15, 14, 13, 12, 11, 10, 9, 8, 1, 0])
   
#     p.savefig('figures/' + feat + '_nhood_clusters_tukeyplot_AD_GM.tiff', format = 'tiff')
#     p.savefig('figures/' + feat + '_nhood_clusters_tukeyplot_AD_GM.svg', format = 'svg')
    
    
    
#     plt.figure(figsize=(10, 6))
#     for i, array in enumerate(group_data):
#         sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

#     plt.xlabel("Value")
#     plt.ylabel("Density")
#     plt.title("Density Distributions")
#     plt.legend()
#     plt.show()
    
    
    
    
    







# NEIGHBOR GM AD (only abeta associated clusters)

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# load in sample all cells adata

In [ ]:

adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')



### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('MG_neighborhood/sub_obj_with_spatial_clustering.h5ad')
adata = adata[adata.obs.spatial_clustering.isin(['0','15','9'])]

# adata = adata[adata.obs.leiden != 'default']

# new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']



# change_cluster_names_dict = {'MG_0': 'Microglia 1',
#          'MG_1': 'Microglia 2',
#          'MO_MAC_0': 'Monocytes',
#          'MO_MAC_1': 'PVM',
#          'MO_MAC_2': 'BLM',
#          'default': 'default'
# }


# cMapDict = {'Microglia 1': '#e377c2',
#  'Microglia 2': '#b5bd61',
#  'Monocytes': '#d62728',
#  'PVM': '#17becf',
#  'BLM': '#ff7f0e',
#  'default': '#8c564b'
# }


# colors_in_order = [cMapDict[i] for i in new_name_order]
# cMapDict = dict(zip(new_name_order, colors_in_order))

# adata.obs['leiden'] = [change_cluster_names_dict[i] for i in adata.obs['leiden']]

# adata.obs['leiden'] = pd.Categorical(adata.obs['leiden'], categories=new_name_order, ordered=False)


cMapDict  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }

adata_meta = adata.obs



# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

# bar charts

In [ ]:

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_Oligodendrocytes', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'Oligos', 'dense aBeta', 'diffuse aBeta', 'all aBeta']
clustering_column = 'spatial_clustering'

for i,feat in enumerate(features):
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    

    dt = feat + '_um'
    ss = adata
    BINS  =[0,5,10,15,20,30,50,75,100,200,500]
    out = pd.cut(ss.obs[dt],bins=BINS, right=True)
    ss.obs['Quantile'] = out.values
    
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_abeta_assoc_nhood_clusters_violin_AD_GM.tiff')
    sc.pl.violin(ss, feat + '_um', groupby=clustering_column, ylabel=('distance to the nearest ' + feature_names[i] + ' (µm)'), palette = cMapDict, save= feat + '_abeta_assoc_nhood_clusters_spatial_clusters_violin_AD_GM.svg')
   

    ax = plt.figure(figsize=(8, 6))

    matpl=sc.pl.matrixplot(ss, var_names = adata.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
    matpl.legend(show=False)


    import numpy as np
    sns.set_style("whitegrid", {'axes.grid' : False})




    oDF = ss.obs.groupby(['Quantile',clustering_column,'origfilenum']).size().T.reset_index().groupby(['Quantile',clustering_column]).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
    oDFPct = 100*oDF/oDF.sum(axis=0)

    oDFPct.T.plot(kind='bar',stacked=True,color=cMapDict, legend=True, )
    plt.legend(title='Cluster ID',loc=(1,0.0))
    plt.xlabel('Grouped distance to the nearest ' + feature_names[i] + ' (µm)')
    xtick_labels = [tick.get_text() for tick in plt.gca().get_xticklabels()]
    #plt.gca().set_xticklabels(['[0]']+xtick_labels[1:],rotation = 45)
    modded_labels = [s.replace(']', ')') for s in xtick_labels]
    plt.gca().set_xticklabels(modded_labels,rotation = 45)
    



    # oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

    # oDFPct = oDF.T#/oDF.sum(axis=1)
    # oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
    # ax[1].set_ylabel('% of cells that \n are Microglial clusters')
    # ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
    # ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
    # ax[1].set_ylim(0,40)

    plt.tight_layout()
    plt.savefig('figures/' + feat + '_abeta_assoc_nhood_clusters_barplot_AD_GM.tiff', format = 'tiff')
    plt.savefig('figures/' + feat + '_abeta_assoc_nhood_clusters_barplot_AD_GM.svg', format = 'svg')


# stats testing

In [ ]:
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

clustering_column = 'spatial_clustering'



tukeyresvals = []
anova_pvals = []
anova_test_stats = []


for i,feat in enumerate(features):
    
    
    
    #1 px=0.3774um
    
    adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
    dt = feat + '_um'
    ss = adata
    # ss = adata[(adata.obs.ImageType =='AD')]
    # ss = ss[ss.obs.Parent=='Grey Matter']
    
    anova_table = ss.obs[[clustering_column, feat + '_um']]
    
    groups = anova_table[clustering_column].unique()
    
    
    group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
    group_dict = dict(zip(groups, group_data))
    
    means = [np.mean(i) for i in group_data]
    meantable = pd.DataFrame({'names':groups, 'means':means})
    lowest_mean_group = groups[means.index(np.min(means))]
    
    
    f_statistic, p_value = f_oneway(*group_data)
    print(feat)
    print(p_value)
    
    tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
    print(tukey_results)
    
    tukeyresvals.append(tukey_results)
    anova_pvals.append(p_value)
    anova_test_stats.append(f_statistic)
    
    
    p = plot_simultaneous_custom(tukey_results, comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um', neworder = [1,2,0])

    p.savefig('figures/' + feat + '_abeta_assoc_nhood_clusters_tukeyplot_AD_GM.tiff', format = 'tiff')
    p.savefig('figures/' + feat + '_abeta_assoc_nhood_clusters_tukeyplot_AD_GM.svg', format = 'svg')



    plt.figure(figsize=(10, 6))
    for i, array in enumerate(group_data):
        sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Density Distributions")
    plt.legend()
    plt.show()

adjusted_anova_pvals = multipletests(anova_pvals, method = 'fdr_bh')[1]

for i,feat in enumerate(features):
    tukey_r = tukeyresvals[i]
    
    tukey_dframe = pd.DataFrame(tukey_r.summary())
    tukey_dframe.columns = tukey_dframe.iloc[0,:]
    tukey_dframe = tukey_dframe.iloc[1:,:]
    tukey_dframe['ANOVA p-value'] = anova_pvals[i]
    tukey_dframe['ANOVA p-value adj.'] = adjusted_anova_pvals[i]
    tukey_dframe['ANOVA F test statistic'] = anova_test_stats[i]
    
    tukey_dframe.to_csv('stat_tables/' + feat + '_abeta_assoc_nhood_clusters_tukeyplot_AD_GM.csv')
                        
    
    
    
    
    
    

In [ ]:


# import pandas as pd
# from scipy.stats import f_oneway
# from statsmodels.stats.multicomp import pairwise_tukeyhsd

# features = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta', 'fabseg_distance_to_nearest_Astrocytes', 'fabseg_distance_to_nearest_Neurons', 'fabseg_distance_to_nearest_dense_aBeta', 'fabseg_distance_to_nearest_diffuse_aBeta', 'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']
# feature_names = ['custom_vasc', 'custom_apoe', 'custom_aBeta', 'Astrocytes', 'Neurons', 'dense aBeta', 'diffuse aBeta', 'all aBeta']

# clustering_column = 'spatial_clustering'

# for i,feat in enumerate(features):
    
    
    
#     #1 px=0.3774um
    
#     adata.obs[feat + '_um'] = adata.obs[feat]*0.3774
    
#     dt = feat + '_um'
#     ss = adata
#     # ss = adata[(adata.obs.ImageType =='AD')]
#     # ss = ss[ss.obs.Parent=='Grey Matter']
    
#     anova_table = ss.obs[[clustering_column, feat + '_um']]
    
#     groups = anova_table[clustering_column].unique()
    
    
#     group_data = [anova_table[anova_table[clustering_column] == g][feat + '_um'] for g in groups]
#     group_dict = dict(zip(groups, group_data))
    
#     means = [np.mean(i) for i in group_data]
#     meantable = pd.DataFrame({'names':groups, 'means':means})
#     lowest_mean_group = groups[means.index(np.min(means))]
    
#     f_statistic, p_value = f_oneway(*group_data)
#     print(feat)
#     print(p_value)
    
#     tukey_results = pairwise_tukeyhsd(anova_table[feat + '_um'], anova_table[clustering_column])
#     print(tukey_results)
    
#     p = plot_simultaneous_custom(tukey_results, comparison_name=lowest_mean_group, figsize=(7,5), xlabel=feat + '_um', neworder = [1,2,0])

#     p.savefig('figures/' + feat + '_abeta_assoc_nhood_clusters_tukeyplot_AD_GM.tiff', format = 'tiff')
#     p.savefig('figures/' + feat + '_abeta_assoc_nhood_clusters_tukeyplot_AD_GM.svg', format = 'svg')
    
    
    
#     plt.figure(figsize=(10, 6))
#     for i, array in enumerate(group_data):
#         sns.kdeplot(array, label=f'Distribution {i+1}', linewidth=2)

#     plt.xlabel("Value")
#     plt.ylabel("Density")
#     plt.title("Density Distributions")
#     plt.legend()
#     plt.show()
    
    
    
    
    







In [ ]:


# merge data frames

stardist_data_sub = stardist_data[stardist_data['phenotype'] != 'Microglia']
stardist_data_sub = stardist_data_sub[['spatial_X', 'spatial_Y', 'Parent', 'filenum', 'phenotype']]

fig7_data_sub = fig7_data.obs[['spatial_X', 'spatial_Y', 'Parent', 'ImageID', clustering_column]]
fig7_data_sub.columns = ['spatial_X', 'spatial_Y', 'Parent', 'filenum', 'phenotype']


merged_df = pd.concat([stardist_data_sub, non_nuc_df, fig7_data_sub])


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

merged_df['disease'] = [ad_cnt_dict[i] for i in merged_df['filenum']]



# create paula ver.

stardist_data_sub = stardist_data
stardist_data_sub = stardist_data_sub[['spatial_X', 'spatial_Y', 'Parent', 'filenum', 'phenotype']]

fig7_data_sub = fig7_data.obs[['spatial_X', 'spatial_Y', 'Parent', 'ImageID', clustering_column]]
fig7_data_sub.columns = ['spatial_X', 'spatial_Y', 'Parent', 'filenum', 'phenotype']


merged_df_noOTSM = pd.concat([stardist_data_sub, non_nuc_df])


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

merged_df_noOTSM['disease'] = [ad_cnt_dict[i] for i in merged_df_noOTSM['filenum']]


## adjust to stardist sizes, subset to chosen images

In [ ]:
images_to_view = ['3026', '3155']
sub_dfs_to_merge = []


for image in images_to_view:
    
    sub_merge = merged_df[merged_df['filenum'] == image]
    stardist_subset = stardist_data[stardist_data['filenum'] == image]
    
    minx_star = min(stardist_subset['spatial_X'])
    maxx_star = max(stardist_subset['spatial_X'])
    
    miny_star = min(stardist_subset['spatial_Y'])
    maxy_star = max(stardist_subset['spatial_Y'])
    
    sub_merge_2 = sub_merge[(sub_merge['spatial_X'] >= minx_star) & (sub_merge['spatial_X'] <= maxx_star) & (sub_merge['spatial_Y'] >= miny_star) & (sub_merge['spatial_Y'] <= maxy_star)]
    sub_dfs_to_merge.append(sub_merge_2)

    
    
merged_df_2 = pd.concat(sub_dfs_to_merge)



## adjust other df to stardist sizes, subset to chosen images

In [ ]:
images_to_view = ['3026', '3155']
sub_dfs_to_merge = []


for image in images_to_view:
    
    sub_merge = merged_df_noOTSM[merged_df_noOTSM['filenum'] == image]
    stardist_subset = stardist_data[stardist_data['filenum'] == image]
    
    minx_star = min(stardist_subset['spatial_X'])
    maxx_star = max(stardist_subset['spatial_X'])
    
    miny_star = min(stardist_subset['spatial_Y'])
    maxy_star = max(stardist_subset['spatial_Y'])
    
    sub_merge_2 = sub_merge[(sub_merge['spatial_X'] >= minx_star) & (sub_merge['spatial_X'] <= maxx_star) & (sub_merge['spatial_Y'] >= miny_star) & (sub_merge['spatial_Y'] <= maxy_star)]
    sub_dfs_to_merge.append(sub_merge_2)

    
    
merged_df_2_noOTSM = pd.concat(sub_dfs_to_merge)



## plot merged data

In [ ]:

images_to_view = ['3026', '3155']

for image in images_to_view:
    sub_merge = merged_df_2[merged_df_2['filenum'] == image]
    
    
    
    
#     # Automatically assign colors to unique categories
#     unique_categories = sub_merge['phenotype'].unique()
#     palette = sns.color_palette("husl", len(unique_categories))
#     color_dict = dict(zip(unique_categories, palette))

#     # Create a scatter plot
#     sns.scatterplot(data=sub_merge, x='spatial_X', y='spatial_Y', hue='phenotype', palette=color_dict, s=100, alpha=0.7, size=0)

#     # Set axis labels and title
#     plt.xlabel('X-axis')
#     plt.ylabel('Y-axis')
#     plt.title('Scatter Plot with Points Colored by Category')

#     # Show the legend
#     plt.legend(title='Category', bbox_to_anchor=(1.04, 1), loc='upper left')

#     # Show the plot
#     plt.tight_layout()
#     plt.show()
    
    
    
    fig,ax=plt.subplots(1,1,figsize=(10,8))
    sns.set_style('white')
    ss = sub_merge

    sns.scatterplot(data=ss,x='spatial_X',y='spatial_Y',hue='phenotype', 
                    palette=get_colors(len(set(ss['phenotype']))), s=2)
    ax.invert_yaxis()
    plt.axis('off')
    ax.legend(title='')
    plt.tight_layout()
    plt.show()
    
    
    
    fig,ax=plt.subplots(1,1,figsize=(10,8))
    sns.set_style('white')
    ss = sub_merge

    sns.scatterplot(data=ss,x='spatial_X',y='spatial_Y',hue='Parent', 
                    palette=get_colors(len(set(ss['Parent'])) - 1), s=2)
    ax.invert_yaxis()
    plt.axis('off')
    ax.legend(title='')
    plt.tight_layout()
    plt.show()
    
    
    

In [ ]:
# create additional dataframes for each parent

merged_df_WM = merged_df[merged_df['Parent'] == 'White Matter']
merged_df_GM = merged_df[merged_df['Parent'] == 'Grey Matter']

merged_df_2_WM = merged_df_2[merged_df_2['Parent'] == 'White Matter']
merged_df_2_GM = merged_df_2[merged_df_2['Parent'] == 'Grey Matter']



merged_df_WM_noOTSM = merged_df_noOTSM[merged_df_noOTSM['Parent'] == 'White Matter']
merged_df_GM_noOTSM = merged_df_noOTSM[merged_df_noOTSM['Parent'] == 'Grey Matter']

merged_df_2_WM_noOTSM = merged_df_2_noOTSM[merged_df_2_noOTSM['Parent'] == 'White Matter']
merged_df_2_GM_noOTSM = merged_df_2_noOTSM[merged_df_2_noOTSM['Parent'] == 'Grey Matter']

In [ ]:
# clean up data frames 

dfs = [merged_df_2, merged_df, merged_df_WM, merged_df_GM, merged_df_2_WM, merged_df_2_GM, merged_df_WM_noOTSM, merged_df_GM_noOTSM, merged_df_2_WM_noOTSM, merged_df_2_GM_noOTSM, merged_df_noOTSM, merged_df_2_noOTSM]
df_names = ['merged_df_2.pkl', 'merged_df.pkl', 'merged_df_WM.pkl', 'merged_df_GM.pkl', 'merged_df_2_WM.pkl', 'merged_df_2_GM.pkl', 'merged_df_WM_noOTSM.pkl', 'merged_df_GM_noOTSM.pkl', 'merged_df_2_WM_noOTSM.pkl', 'merged_df_2_GM_noOTSM.pkl', 'merged_df_noOTSM.pkl', 'merged_df_2_noOTSM.pkl']



for i, df in enumerate(dfs):
    print(collections.Counter(df['Parent']))
    df.index = range(len(df))
    df.to_pickle(df_names[i])
    
    


df_to_vis = merged_df_2

for file in set(df_to_vis['filenum']):
    subdf = df_to_vis[df_to_vis['filenum'] == file]
    for cell in set(subdf['phenotype']):
    
        cellsub_df = subdf[subdf['phenotype'] == cell]

        fig,ax=plt.subplots(1,1,figsize=(10,8))
        sns.set_style('white')
        ss = cellsub_df

        sns.scatterplot(data=ss,x='spatial_X',y='spatial_Y',hue='phenotype', 
                        palette=get_colors(len(set(ss['phenotype']))), s=2)
        ax.invert_yaxis()
        ax.legend(title=cell)
        plt.tight_layout()
        plt.show()


In [ ]:
df

In [ ]:
# save merged datasets

merged_df_2.to_pickle('merged_df_2.pkl')
merged_df.to_pickle('merged_df.pkl')



In [ ]:
https://github.com/nolanlab/NeighborhoodCoordination